# Day 22 — Data Drift Simulation & Monitoring Foundations

**Goal:** Synthetically break the reference distribution three ways, calculate drift metrics (PSI, KS), and demonstrate that the champion model degrades under concept drift.

| Drift Type | What changes | Simulation |
|---|---|---|
| **Feature Drift** | Input distribution shifts | LIMIT_BAL ×1.5, AGE +5 yrs |
| **Target Drift** | Default rate spikes | Recession scenario (22% → 40%) |
| **Concept Drift** | Feature→target relationship breaks | PAY_0 loses predictive power |

The simulated dataset is saved to `data/simulated_production_data.csv` for use in Day 23 monitoring.

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore")

# Resolve project root regardless of whether the notebook is launched from
# the project root or from inside the notebooks/ subdirectory.
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)                     # make relative paths work everywhere
sys.path.insert(0, str(PROJECT_ROOT))      # allow  `from src.x import y`

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print(f"Project root : {PROJECT_ROOT}")
print(f"Working dir  : {Path.cwd()}")

## 1. Load Reference Data

The **reference** (a.k.a. baseline) dataset is the held-out test set the model was validated on. In production, this is what we compare incoming data against.

In [ ]:
PROCESSED_PATHS = [
    PROJECT_ROOT / "data/processed/credit_default_eda_ready.parquet",
    PROJECT_ROOT / "data/processed/credit_default_validated.parquet",
    PROJECT_ROOT / "data/processed/credit_default_cleaned.parquet",
]

df = None
for p in PROCESSED_PATHS:
    if p.exists():
        df = pd.read_parquet(p)
        print(f"Loaded {len(df):,} rows from {p.name}")
        break

if df is None:
    print("No processed parquet found — downloading and processing UCI dataset (~30 s)...")
    from src.data_ingestion import load_data
    df = load_data()
    print(f"Downloaded and processed {len(df):,} rows")

from sklearn.model_selection import train_test_split

TARGET = "default"
feat_cols = [c for c in df.columns if c != TARGET]
_, X_test, _, y_test = train_test_split(
    df[feat_cols], df[TARGET], test_size=0.2, random_state=42, stratify=df[TARGET]
)

ref_df = X_test.copy()
ref_df[TARGET] = y_test.values

print(f"\nReference set : {len(ref_df):,} rows")
print(f"Default rate  : {ref_df[TARGET].mean():.1%}")
print(f"\nKey feature stats:")
print(ref_df[["LIMIT_BAL", "AGE", "PAY_0"]].describe().round(1))

## 2. PSI Helper Function

**Population Stability Index (PSI)** measures how much a distribution has shifted between two datasets.

| PSI | Interpretation | Action |
|---|---|---|
| < 0.10 | No significant change | Monitor |
| 0.10 – 0.25 | Moderate shift | Investigate |
| > 0.25 | Major shift | Retrain model |

In [ ]:
def calculate_psi(reference: pd.Series, production: pd.Series, bins: int = 10) -> float:
    """Population Stability Index — quantifies distribution shift."""
    ref_clean  = reference.dropna()
    prod_clean = production.dropna()

    _, edges = np.histogram(ref_clean, bins=bins)
    edges[0]  -= 1e-6
    edges[-1] += 1e-6

    ref_freq,  _ = np.histogram(ref_clean,  bins=edges)
    prod_freq, _ = np.histogram(prod_clean, bins=edges)

    ref_pct  = ref_freq  / len(ref_clean)
    prod_pct = prod_freq / len(prod_clean)

    ref_pct  = np.where(ref_pct  == 0, 1e-6, ref_pct)
    prod_pct = np.where(prod_pct == 0, 1e-6, prod_pct)

    return float(np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct)))


def psi_status(val: float) -> str:
    if val < 0.10: return "✅  No shift      (PSI < 0.10)"
    if val < 0.25: return "⚠️   Investigate   (0.10 ≤ PSI < 0.25)"
    return           "🚨  ACTION NEEDED  (PSI ≥ 0.25)"


# Sanity check: PSI of identical distributions should be ≈ 0
psi_self = calculate_psi(ref_df["LIMIT_BAL"], ref_df["LIMIT_BAL"])
print(f"PSI(LIMIT_BAL vs itself) = {psi_self:.6f}  ← expected ≈ 0")

## 3. Simulate Feature Drift

**Scenario:** Economic boom raises credit limits across the board; applicant pool also skews older.

- `LIMIT_BAL` scaled up 50% with Gaussian noise
- `AGE` shifted right by ~5 years
- 10% of `AGE` values set to `NaN` (upstream system regression)

In [ ]:
np.random.seed(42)
prod_feat = ref_df.copy()

# ── Drift 1: credit limit increase (economic boom) ─────────────────────────
prod_feat["LIMIT_BAL"] = (
    prod_feat["LIMIT_BAL"] * 1.5
    + np.random.normal(0, 5_000, len(prod_feat))
).clip(lower=1_000)

# ── Drift 2: applicant pool ages by ~5 years ───────────────────────────────
prod_feat["AGE"] = (
    prod_feat["AGE"] + np.random.normal(5, 3, len(prod_feat))
).clip(18, 100).round().astype(int)

# ── Data quality drift: 10% missing AGE ────────────────────────────────────
missing_idx = prod_feat.sample(frac=0.10, random_state=42).index
prod_feat.loc[missing_idx, "AGE"] = np.nan

# ── Plot side-by-side distributions ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col in zip(axes, ["LIMIT_BAL", "AGE"]):
    ax.hist(ref_df[col].dropna(),    bins=30, alpha=0.65, density=True,
            color="#2196F3", label="Reference (test set)")
    ax.hist(prod_feat[col].dropna(), bins=30, alpha=0.65, density=True,
            color="#F44336", label="Production (drifted)")
    psi_val = calculate_psi(ref_df[col], prod_feat[col])
    ax.set_title(f"{col}   PSI = {psi_val:.3f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value"); ax.set_ylabel("Density")
    ax.legend(fontsize=9)

plt.suptitle("Feature Drift — Reference vs. Production", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "25_feature_drift_distributions.png", bbox_inches="tight")
plt.show()
print("Saved: 25_feature_drift_distributions.png")

In [ ]:
# ── PSI report ─────────────────────────────────────────────────────────────
psi_limit   = calculate_psi(ref_df["LIMIT_BAL"], prod_feat["LIMIT_BAL"])
psi_age     = calculate_psi(ref_df["AGE"],       prod_feat["AGE"])
missing_pct = prod_feat["AGE"].isna().mean()

print("Feature Drift Report")
print("=" * 62)
print(f"  LIMIT_BAL  PSI = {psi_limit:.3f}   {psi_status(psi_limit)}")
print(f"  AGE        PSI = {psi_age:.3f}   {psi_status(psi_age)}")
print(f"  AGE null%      = {missing_pct:.1%}   "
      f"{'🚨  ACTION NEEDED  (> 5% missing)' if missing_pct > 0.05 else '✅  OK'}")
print()
print("Trigger: PSI > 0.25 → retrain required (docs/threshold_policy.md)")

## 4. Simulate Target Drift

**Scenario:** A sudden recession causes the default rate to spike from 22% to 40%.
This is a leading indicator — if target drift is detected early enough, a bank can tighten thresholds *before* model performance collapses.

In [ ]:
np.random.seed(42)
prod_target = ref_df.copy()

RECESSION_RATE = 0.40
ref_rate = prod_target[TARGET].mean()
n_extra  = int((RECESSION_RATE - ref_rate) * len(prod_target))

non_default_idx = prod_target[prod_target[TARGET] == 0].index
flip_idx = np.random.choice(non_default_idx, size=n_extra, replace=False)
prod_target.loc[flip_idx, TARGET] = 1

prod_rate = prod_target[TARGET].mean()
ks_stat, ks_p = stats.ks_2samp(ref_df[TARGET], prod_target[TARGET])

print(f"Reference default rate  : {ref_rate:.1%}")
print(f"Production default rate : {prod_rate:.1%}   (+{prod_rate - ref_rate:.1%})")
print(f"KS statistic            : {ks_stat:.4f}")
print(f"KS p-value              : {ks_p:.2e}")
print(f"Target drift detected   : {'YES 🚨' if ks_p < 0.05 else 'No ✅'}")

# ── Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
categories = ["No Default (0)", "Default (1)"]
x = np.arange(len(categories))
w = 0.38

ref_props  = ref_df[TARGET].value_counts(normalize=True).sort_index().values
prod_props = prod_target[TARGET].value_counts(normalize=True).sort_index().values

ax.bar(x - w/2, ref_props,  w, label="Reference",           color="#2196F3", alpha=0.85)
ax.bar(x + w/2, prod_props, w, label="Production (recession)", color="#F44336", alpha=0.85)

for xi, (rv, pv) in zip(x, zip(ref_props, prod_props)):
    ax.text(xi - w/2, rv + 0.01, f"{rv:.1%}", ha="center", fontsize=9, color="#2196F3", fontweight="bold")
    ax.text(xi + w/2, pv + 0.01, f"{pv:.1%}", ha="center", fontsize=9, color="#F44336", fontweight="bold")

ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylabel("Proportion"); ax.set_ylim(0, 1.0)
ax.set_title("Target Drift — Default Rate Shift (Recession Scenario)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "26_target_drift.png", bbox_inches="tight")
plt.show()
print("Saved: 26_target_drift.png")

## 5. Simulate Concept Drift

**Scenario:** A regulatory change introduces a 60-day payment grace period. Customers now accumulate `PAY_0 ≥ 2` (delay codes) without actually being at elevated default risk. The *feature still exists* — but its relationship to the target has broken.

This is the most dangerous drift type because PSI on `PAY_0` may look normal, yet model AUC collapses.

In [ ]:
np.random.seed(42)
prod_concept = ref_df.copy()

# Corrupt PAY_0 for defaulters: their payment codes now look like good payers
defaulter_mask = prod_concept[TARGET] == 1
prod_concept.loc[defaulter_mask, "PAY_0"] = np.random.choice(
    [-2, -1, 0], size=defaulter_mask.sum(), replace=True
)

# Correlation check: does PAY_0 still predict default?
corr_ref     = ref_df[["PAY_0", TARGET]].corr().loc["PAY_0", TARGET]
corr_concept = prod_concept[["PAY_0", TARGET]].corr().loc["PAY_0", TARGET]

print(f"Corr(PAY_0, default) — Reference     : {corr_ref:+.4f}")
print(f"Corr(PAY_0, default) — Concept-drifted: {corr_concept:+.4f}")
print(f"Signal loss: {abs(corr_ref - corr_concept) / abs(corr_ref):.1%}")
print()

# NOTE: PSI on PAY_0 looks fine (the feature distribution barely changes)
# because defaulters' codes move from {2,3,...} to {-2,-1,0} — still valid codes
psi_pay0_ref     = calculate_psi(ref_df["PAY_0"], prod_concept["PAY_0"])
print(f"PSI(PAY_0) = {psi_pay0_ref:.3f}  — {psi_status(psi_pay0_ref)}")
print("^ PSI alone would NOT catch this drift — correlation monitoring is required")

In [ ]:
# ── Show AUC degradation with the loaded model ──────────────────────────────
from src.features.engineer import engineer_features

MODEL_PATH = PROJECT_ROOT / "models" / "champion_model.joblib"
META_PATH  = PROJECT_ROOT / "models" / "champion_model_metadata.json"

if MODEL_PATH.exists() and META_PATH.exists():
    import joblib
    from sklearn.metrics import roc_auc_score

    model        = joblib.load(MODEL_PATH)
    feature_names = json.loads(META_PATH.read_text()).get("feature_names", [])

    def score_df(raw_df):
        X_eng = engineer_features(raw_df.drop(columns=[TARGET]))
        X = X_eng[feature_names] if feature_names else X_eng
        return model.predict_proba(X)[:, 1]

    auc_ref     = roc_auc_score(ref_df[TARGET],     score_df(ref_df))
    auc_concept = roc_auc_score(prod_concept[TARGET], score_df(prod_concept))

    RETRAIN_TRIGGER = 0.72
    print(f"AUC — Reference         : {auc_ref:.4f}")
    print(f"AUC — Concept-drifted   : {auc_concept:.4f}")
    print(f"Drop                    : {auc_ref - auc_concept:.4f}  ({(auc_ref - auc_concept)/auc_ref:.1%} relative)")
    print(f"Retrain trigger (< {RETRAIN_TRIGGER}) : {'TRIGGERED 🚨' if auc_concept < RETRAIN_TRIGGER else 'Not triggered'}")

    # Plot
    fig, ax = plt.subplots(figsize=(7, 4))
    aucs   = [auc_ref, auc_concept]
    labels = ["Reference\n(test set)", "Concept-drifted\n(PAY_0 corrupted)"]
    colors = ["#2196F3", "#F44336"]
    bars = ax.bar(labels, aucs, color=colors, alpha=0.85, width=0.5)
    ax.axhline(RETRAIN_TRIGGER, color="orange", linestyle="--", linewidth=1.5,
               label=f"Retrain trigger (AUC < {RETRAIN_TRIGGER})")
    ax.set_ylim(0.50, 0.85)
    ax.set_ylabel("AUC-ROC")
    ax.set_title("Concept Drift — AUC Degradation", fontsize=12, fontweight="bold")
    for bar, val in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "27_concept_drift_auc.png", bbox_inches="tight")
    plt.show()
    print("Saved: 27_concept_drift_auc.png")
else:
    print("Model not found — run `python main.py --download` first.")
    print(f"Correlation drop shown above is sufficient to demonstrate concept drift.")

## 6. Summary & Save Production Dataset

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
print("=" * 70)
print("  DRIFT MONITORING SUMMARY — Day 22")
print("=" * 70)
print(f"{'Drift Type':<18} {'Feature':<15} {'Metric':<22} Status")
print("-" * 70)
print(f"{'Feature Drift':<18} {'LIMIT_BAL':<15} {f'PSI = {psi_limit:.3f}':<22} {psi_status(psi_limit)}")
print(f"{'Feature Drift':<18} {'AGE':<15} {f'PSI = {psi_age:.3f}':<22} {psi_status(psi_age)}")
print(f"{'Data Quality':<18} {'AGE missing':<15} {f'{missing_pct:.1%} null':<22} {'🚨  > 5% threshold' if missing_pct > 0.05 else '✅  OK'}")
print(f"{'Target Drift':<18} {'default rate':<15} {f'KS p = {ks_p:.2e}':<22} {'🚨  p < 0.05' if ks_p < 0.05 else '✅  OK'}")
print(f"{'Concept Drift':<18} {'PAY_0 signal':<15} {'Corr drop':<22} {'🚨  Correlation collapsed'}")
print()
print("Thresholds from docs/threshold_policy.md:")
print("  PSI > 0.25      → retrain")
print("  AUC < 0.72      → retrain")
print("  Approval ± 10pp → investigate")

# ── Save drifted dataset for Day 23 ────────────────────────────────────────
out_path = PROJECT_ROOT / "data" / "simulated_production_data.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
prod_feat.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")
print(f"Shape : {prod_feat.shape}")
print(f"Default rate : {prod_feat[TARGET].mean():.1%}  (reference was {ref_rate:.1%})")